# 05 Model Comparison and Error Analysis

This notebook compares the **TF-IDF + Logistic Regression** baseline against the **BioBERT** transformer model using the saved output files.


## Purpose

Use this notebook after training to answer the key research questions:

- Did BioBERT improve over the classical baseline?
- On which kinds of examples does BioBERT help?
- Where do both models still fail?
- What examples are useful for qualitative discussion in the thesis/report?


In [1]:
from pathlib import Path
import json

import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report


In [2]:
PROJECT_ROOT = Path(r'C:\Users\ribam\Desktop\Reseach\Dataset')
OUTPUT_ROOT = PROJECT_ROOT / 'output'
COMPARE_DIR = OUTPUT_ROOT / 'model_comparison'
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

TFIDF_METRICS_PATH = OUTPUT_ROOT / 'tfidf_logreg_metrics.json'
TFIDF_PRED_PATH = OUTPUT_ROOT / 'tfidf_logreg_test_predictions.csv'

BIOBERT_DIR = OUTPUT_ROOT / 'biobert_fakehealth_healthfact'
BIOBERT_METRICS_PATH = BIOBERT_DIR / 'biobert_metrics.json'
BIOBERT_PRED_PATH = BIOBERT_DIR / 'biobert_test_predictions.csv'

TFIDF_METRICS_PATH, BIOBERT_METRICS_PATH


(WindowsPath('C:/Users/ribam/Desktop/Reseach/Dataset/output/tfidf_logreg_metrics.json'),
 WindowsPath('C:/Users/ribam/Desktop/Reseach/Dataset/output/biobert_fakehealth_healthfact/biobert_metrics.json'))

In [3]:
with open(TFIDF_METRICS_PATH, 'r', encoding='utf-8') as f:
    tfidf_metrics = json.load(f)

with open(BIOBERT_METRICS_PATH, 'r', encoding='utf-8') as f:
    biobert_metrics = json.load(f)

tfidf_metrics, biobert_metrics


({'model': 'TF-IDF + Logistic Regression',
  'train_rows': 9588,
  'dev_rows': 1332,
  'test_rows': 1311,
  'dev_metrics': {'split': 'dev',
   'accuracy': 0.683933933933934,
   'precision': 0.7755960729312763,
   'recall': 0.6793611793611793,
   'f1': 0.7242960052390308},
  'test_metrics': {'split': 'test',
   'accuracy': 0.6872616323417239,
   'precision': 0.7713458755426917,
   'recall': 0.6789808917197452,
   'f1': 0.7222222222222222}},
 {'model_name': 'dmis-lab/biobert-base-cased-v1.1',
  'max_length': 256,
  'num_epochs': 5,
  'train_batch_size': 4,
  'eval_batch_size': 4,
  'gradient_accumulation_steps': 2,
  'learning_rate': 2e-05,
  'device': 'cpu',
  'train_rows': 9588,
  'dev_rows': 1332,
  'test_rows': 1311,
  'dev_metrics': {'eval_loss': 0.4946806728839874,
   'eval_accuracy': 0.7424924924924925,
   'eval_precision': 0.733862959285005,
   'eval_recall': 0.9078624078624079,
   'eval_f1': 0.8116419549697969,
   'eval_runtime': 121.7321,
   'eval_samples_per_second': 10.942,
 

In [4]:
comparison_df = pd.DataFrame([
    {
        'model': 'TF-IDF + Logistic Regression',
        'dev_accuracy': tfidf_metrics['dev_metrics']['accuracy'],
        'dev_precision': tfidf_metrics['dev_metrics']['precision'],
        'dev_recall': tfidf_metrics['dev_metrics']['recall'],
        'dev_f1': tfidf_metrics['dev_metrics']['f1'],
        'test_accuracy': tfidf_metrics['test_metrics']['accuracy'],
        'test_precision': tfidf_metrics['test_metrics']['precision'],
        'test_recall': tfidf_metrics['test_metrics']['recall'],
        'test_f1': tfidf_metrics['test_metrics']['f1'],
    },
    {
        'model': 'BioBERT',
        'dev_accuracy': biobert_metrics['dev_metrics']['eval_accuracy'],
        'dev_precision': biobert_metrics['dev_metrics']['eval_precision'],
        'dev_recall': biobert_metrics['dev_metrics']['eval_recall'],
        'dev_f1': biobert_metrics['dev_metrics']['eval_f1'],
        'test_accuracy': biobert_metrics['test_metrics']['eval_accuracy'],
        'test_precision': biobert_metrics['test_metrics']['eval_precision'],
        'test_recall': biobert_metrics['test_metrics']['eval_recall'],
        'test_f1': biobert_metrics['test_metrics']['eval_f1'],
    },
])

comparison_df.round(4)


,model,dev_accuracy,dev_precision,dev_recall,dev_f1,test_accuracy,test_precision,test_recall,test_f1
0,TF-IDF + Logistic Regression,0.6839,0.7756,0.6794,0.7243,0.6873,0.7713,0.6790,0.7222
1,BioBERT,0.7425,0.7339,0.9079,0.8116,0.7338,0.7290,0.8841,0.7991


In [5]:
comparison_df.to_csv(COMPARE_DIR / 'model_metric_comparison.csv', index=False)
print('Saved metric comparison to', COMPARE_DIR / 'model_metric_comparison.csv')


Saved metric comparison to C:\Users\ribam\Desktop\Reseach\Dataset\output\model_comparison\model_metric_comparison.csv


In [6]:
tfidf_pred = pd.read_csv(TFIDF_PRED_PATH)
biobert_pred = pd.read_csv(BIOBERT_PRED_PATH)

tfidf_pred = tfidf_pred.rename(columns={'prediction': 'tfidf_prediction'})
biobert_pred = biobert_pred.rename(columns={'prediction': 'biobert_prediction'})

merged = tfidf_pred.merge(
    biobert_pred[['dataset', 'split', 'record_id', 'label', 'text', 'biobert_prediction', 'prob_class_0', 'prob_class_1']],
    on=['dataset', 'split', 'record_id', 'label', 'text'],
    how='inner',
)

print('Merged rows:', len(merged))
merged.head(3)


Merged rows: 1311


,dataset,split,record_id,text,label,tfidf_prediction,biobert_prediction,prob_class_0,prob_class_1
0,healthfact,test,33456,A mother revealed to her child in a letter aft...,0,0,1,0.320590,0.679410
1,healthfact,test,2542,Study says too many Americans still drink too ...,1,1,1,0.343338,0.656662
2,healthfact,test,26678,Viral image Says 80% of novel coronavirus case...,1,0,1,0.467584,0.532416


In [7]:
merged['tfidf_correct'] = merged['tfidf_prediction'] == merged['label']
merged['biobert_correct'] = merged['biobert_prediction'] == merged['label']
merged['models_agree'] = merged['tfidf_prediction'] == merged['biobert_prediction']

merged['comparison_bucket'] = 'both_wrong'
merged.loc[merged['tfidf_correct'] & merged['biobert_correct'], 'comparison_bucket'] = 'both_correct'
merged.loc[merged['tfidf_correct'] & ~merged['biobert_correct'], 'comparison_bucket'] = 'tfidf_only_correct'
merged.loc[~merged['tfidf_correct'] & merged['biobert_correct'], 'comparison_bucket'] = 'biobert_only_correct'

merged['comparison_bucket'].value_counts()


comparison_bucket
both_correct            492
tfidf_only_correct      409
biobert_only_correct    228
both_wrong              182
Name: count, dtype: int64

In [8]:
bucket_summary = merged['comparison_bucket'].value_counts().rename_axis('bucket').reset_index(name='count')
bucket_summary['percent'] = (bucket_summary['count'] / len(merged) * 100).round(2)
bucket_summary


,bucket,count,percent
0,both_correct,492,37.53
1,tfidf_only_correct,409,31.20
2,biobert_only_correct,228,17.39
3,both_wrong,182,13.88


In [9]:
bucket_summary.to_csv(COMPARE_DIR / 'comparison_bucket_summary.csv', index=False)
merged.to_csv(COMPARE_DIR / 'merged_model_predictions.csv', index=False)
print('Saved merged comparison files to', COMPARE_DIR)


Saved merged comparison files to C:\Users\ribam\Desktop\Reseach\Dataset\output\model_comparison


In [10]:
dataset_bucket = pd.crosstab(merged['dataset'], merged['comparison_bucket'])
label_bucket = pd.crosstab(merged['label'], merged['comparison_bucket'])

display(dataset_bucket)
display(label_bucket)


comparison_bucket,biobert_only_correct,both_correct,both_wrong,tfidf_only_correct
dataset,,,,
fakehealth,94,84,58,88
healthfact,134,408,124,321


comparison_bucket,biobert_only_correct,both_correct,both_wrong,tfidf_only_correct
label,,,,
0,44,103,114,265
1,184,389,68,144


In [11]:
dataset_bucket.to_csv(COMPARE_DIR / 'dataset_bucket_crosstab.csv')
label_bucket.to_csv(COMPARE_DIR / 'label_bucket_crosstab.csv')


## Qualitative Error Analysis

These tables are especially useful for the research write-up because they give concrete examples of where the transformer helps or still fails.


In [12]:
biobert_wins = merged[merged['comparison_bucket'] == 'biobert_only_correct'][[
    'dataset', 'record_id', 'label', 'tfidf_prediction', 'biobert_prediction', 'prob_class_0', 'prob_class_1', 'text'
]].copy()

tfidf_wins = merged[merged['comparison_bucket'] == 'tfidf_only_correct'][[
    'dataset', 'record_id', 'label', 'tfidf_prediction', 'biobert_prediction', 'prob_class_0', 'prob_class_1', 'text'
]].copy()

both_wrong = merged[merged['comparison_bucket'] == 'both_wrong'][[
    'dataset', 'record_id', 'label', 'tfidf_prediction', 'biobert_prediction', 'prob_class_0', 'prob_class_1', 'text'
]].copy()

print('BioBERT only correct:', len(biobert_wins))
print('TF-IDF only correct:', len(tfidf_wins))
print('Both wrong:', len(both_wrong))


BioBERT only correct: 228
TF-IDF only correct: 409
Both wrong: 182


In [13]:
biobert_wins.head(10)


,dataset,record_id,label,tfidf_prediction,biobert_prediction,prob_class_0,prob_class_1,text
2,healthfact,26678,1,0,1,0.467584,0.532416,Viral image Says 80% of novel coronavirus case...
31,healthfact,760,1,0,1,0.324755,0.675245,"Suicide kills one person every 40 seconds, say..."
41,healthfact,13528,1,0,1,0.422483,0.577517,"Heroin comes in the United States ""from the so..."
46,healthfact,12009,1,0,1,0.269034,0.730966,One in 10 babies born in this country is born ...
49,healthfact,33288,0,1,0,0.779961,0.220039,A few drops of Visine brand eye drops taken in...
52,healthfact,16366,1,0,1,0.216319,0.783681,"Charlie Crist Says Rick Scott signed ""laws req..."
82,healthfact,33983,0,1,0,0.744757,0.255243,NASA and NOAA faked climate data in the GISTEM...
90,healthfact,26657,0,1,0,0.822972,0.177028,The WHO coronavirus test “was a bad test.”
91,healthfact,37950,1,0,1,0.002244,0.997756,"On September 18 2020, Twitter user @JohnCammo ..."
95,healthfact,15867,0,1,0,0.869567,0.130433,Drug Policy Alliance Says Debbie Wasserman Sch...


In [14]:
tfidf_wins.head(10)


,dataset,record_id,label,tfidf_prediction,biobert_prediction,prob_class_0,prob_class_1,text
0,healthfact,33456,0,0,1,0.320590,0.679410,A mother revealed to her child in a letter aft...
3,healthfact,40705,0,0,1,0.372422,0.627578,An email says that 9-year old Craig Shergold o...
8,healthfact,32840,0,0,1,0.338920,0.661080,The media covered up an incident in San Bernar...
9,healthfact,33851,0,0,1,0.468661,0.531339,The mayonnaise oozing from a chicken sandwich ...
11,healthfact,29964,0,0,1,0.298784,0.701216,A film producer claimed actor Kevin Spacey had...
14,healthfact,1356,1,1,0,0.548685,0.451315,Britain backs GSK's gene therapy for 'bubble b...
15,healthfact,29144,0,0,1,0.390815,0.609185,Queen Elizabeth II wore a Burmese Ruby Tiara a...
16,healthfact,933,1,1,0,0.509763,0.490237,South Korea court strikes down abortion law in...
18,healthfact,9960,0,0,1,0.200180,0.799820,Scientists look to stem cells to mend broken h...
22,healthfact,37855,0,0,1,0.320784,0.679216,"United States President Donald Trump tweeted ""..."


In [15]:
both_wrong.head(10)


,dataset,record_id,label,tfidf_prediction,biobert_prediction,prob_class_0,prob_class_1,text
13,healthfact,26248,0,1,1,0.347767,0.652233,“While California is dying … Gavin (Newsom) is...
19,healthfact,11523,0,1,1,0.237030,0.762970,Study suggests a second LDL test
21,healthfact,17064,0,1,1,0.350756,0.649244,A lot of the problems with forest fires ... is...
26,healthfact,10690,0,1,1,0.387929,0.612071,Treating First Time Shoulder Dislocations with...
38,healthfact,11042,0,1,1,0.484330,0.515670,Strobe lighting provides a flicker of hope in ...
40,healthfact,31219,0,1,1,0.416092,0.583908,Accepting a friend request from a stranger wil...
60,healthfact,10970,1,0,0,0.518606,0.481394,"To reverse damage of sitting, take a brisk, ho..."
62,healthfact,30893,0,1,1,0.253949,0.746051,A teenage schoolgirl in Texas became pregnant ...
65,healthfact,3478,1,0,0,0.573844,0.426157,"Lamont fills top public health, insurance agen..."
67,healthfact,5571,1,0,0,0.714376,0.285624,CVI: The impairment affecting children whose e...


In [16]:
biobert_wins.to_csv(COMPARE_DIR / 'biobert_only_correct_examples.csv', index=False)
tfidf_wins.to_csv(COMPARE_DIR / 'tfidf_only_correct_examples.csv', index=False)
both_wrong.to_csv(COMPARE_DIR / 'both_wrong_examples.csv', index=False)
print('Saved example CSVs for qualitative analysis.')


Saved example CSVs for qualitative analysis.


In [17]:
summary = {
    'best_model_by_test_f1': comparison_df.sort_values('test_f1', ascending=False).iloc[0]['model'],
    'tfidf_test_f1': float(comparison_df.loc[comparison_df['model'] == 'TF-IDF + Logistic Regression', 'test_f1'].iloc[0]),
    'biobert_test_f1': float(comparison_df.loc[comparison_df['model'] == 'BioBERT', 'test_f1'].iloc[0]),
    'f1_gain_biobert_over_tfidf': float(
        comparison_df.loc[comparison_df['model'] == 'BioBERT', 'test_f1'].iloc[0] -
        comparison_df.loc[comparison_df['model'] == 'TF-IDF + Logistic Regression', 'test_f1'].iloc[0]
    ),
    'bucket_counts': bucket_summary.set_index('bucket')['count'].to_dict(),
}

(COMPARE_DIR / 'comparison_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
summary


{'best_model_by_test_f1': 'BioBERT',
 'tfidf_test_f1': 0.7222222222222222,
 'biobert_test_f1': 0.7990788716177317,
 'f1_gain_biobert_over_tfidf': 0.07685664939550951,
 'bucket_counts': {'both_correct': 492,
  'tfidf_only_correct': 409,
  'biobert_only_correct': 228,
  'both_wrong': 182}}

## Next Step

After this notebook, the strongest next extension is either:

- **PubMedBERT** as a second domain-specific transformer comparison, or
- a **single-claim inference notebook** for demo/testing with custom user input.
